In [2]:
import pandas as pd
from credit_risk.data.registry import build_default_registry
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

In [3]:
registry = build_default_registry()
accepted_dataset = registry.get('lending_club_accepted_raw')

In [5]:
df = pd.read_csv(accepted_dataset.path)


/var/folders/_y/z6qlgwb57vz2l3q1xqw5zb200000gn/T/ipykernel_14812/3650065708.py:1: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(accepted_dataset.path)


## Creating data dictionary with descriptions

In [ ]:
# Build a column-level data dictionary using Lending Club's official definitions.
# Source: https://resources.lendingclub.com/LCDataDictionary.xlsx
import re

DATA_DICTIONARY_URL = "https://resources.lendingclub.com/LCDataDictionary.xlsx"

loanstats_dict = pd.read_excel(DATA_DICTIONARY_URL, sheet_name="LoanStats")
browsenotes_dict = pd.read_excel(DATA_DICTIONARY_URL, sheet_name="browseNotes")


def normalize_name(name: str) -> str:
    """Normalize dictionary and dataset field names to a common format for robust matching."""
    normalized = str(name).replace("\xa0", " ").strip().lower()
    normalized = re.sub(r"(?<!^)(?=[A-Z])", "_", normalized)
    normalized = normalized.replace(" ", "_").replace("-", "_").replace("/", "_")
    normalized = re.sub(r"[^a-z0-9_]+", "", normalized)
    normalized = re.sub(r"_+", "_", normalized).strip("_")
    return normalized


dictionary_rows = pd.concat(
    [
        loanstats_dict.rename(columns={"LoanStatNew": "dictionary_field", "Description": "description"})[
            ["dictionary_field", "description"]
        ],
        browsenotes_dict.rename(columns={"BrowseNotesFile": "dictionary_field", "Description": "description"})[
            ["dictionary_field", "description"]
        ],
    ],
    ignore_index=True,
)

normalized_description_map = {}
for _, row in dictionary_rows.dropna(subset=["dictionary_field", "description"]).iterrows():
    key = normalize_name(row["dictionary_field"])
    if key not in normalized_description_map:
        normalized_description_map[key] = str(row["description"]).strip()

# Official file uses `verified_status_joint`, while dataset column is `verification_status_joint`.
normalized_description_map["verification_status_joint"] = normalized_description_map.get(
    "verified_status_joint"
)


column_info = pd.DataFrame({"column": df.columns})
column_info["normalized_column"] = column_info["column"].map(normalize_name)
column_info["description"] = column_info["normalized_column"].map(normalized_description_map)

missing_description_columns = column_info.loc[column_info["description"].isna(), "column"].tolist()
if missing_description_columns:
    raise ValueError(
        "Missing official descriptions for columns: " + ", ".join(missing_description_columns)
    )

data_dictionary_df = column_info[["column", "description"]].copy()

print(f"Data dictionary ready: {len(data_dictionary_df)} columns documented.")

data_dictionary_df.head(20)

Data dictionary ready: 151 columns documented.


,column,description
0,id,A unique LC assigned ID for the loan listing.
1,member_id,A unique LC assigned Id for the borrower member.
2,loan_amnt,The listed amount of the loan applied for by t...
3,funded_amnt,The total amount committed to that loan at tha...
4,funded_amnt_inv,The total amount committed by investors for th...
5,term,The number of payments on the loan. Values are...
6,int_rate,Interest Rate on the loan
7,installment,The monthly payment owed by the borrower if th...
8,grade,LC assigned loan grade
9,sub_grade,LC assigned loan subgrade


In [9]:
data_dictionary_df.to_csv('../artifacts/reports/lending_club_accepted_raw_data_dictionary.csv')